# # 종합 실습 가이드_모델개발 및 최적화



- 실습 주제 : 머신러닝 기법을 활용한 은행 고객 이탈 예측 모형 생성
- 실습 목표 : Validation 및 Test 데이터의 평균 AUC 최대화
- 실습 조건
  - 분석 데이타 : 실습에서 사용한 동일한 데이타 (bank_churn_train.csv)
  - Target 변수 : Exited (이탈 여부) *Target 변수는 어떠한 데이타 변환도 하지 않음
  - 학습 조건 :  아래 Baseline Code 를 참고하여 **(필수)** 과정인 데이타 불러오기 및 데이터 분할만 **아래 제시된 코드를 그대로 사용**하고, 그외 모든 과정은 자율적으로 선택
  - 데이타 건수 : 데이타 제거는 하지 않음 (전체/Train/Test 건수 그대로 유지), 중복 데이타가 있어도 제거하지 않음
  - 기타 : 수업에서 다루지 않은 분석 기법 사용도 허용. 예) AutoML
- 실습 제출물
   - 종합실습_결과물_모델개발및최적화_OOO.xlsx 에 실습 결과 작성 후 코드 (.ipynb)와 같이 제출   
   - 실습 결과는 제출 코드로 재현이 가능해야 함
   - 제출시 화일명은 본인 이름으로 2개 화일 제출 (엑셀, 코드)   
     예) **종합실습_결과물_모델개발및최적화_홍길동.xlsx**, **종합실습_결과물_모델개발및최적화_홍길동.ipynb**


# Baseline Code
데이타 불러오기 **(필수)**

In [ ]:
import pandas as pd

df = pd.read_csv("bank_churn_train.csv", encoding="cp949")

Data Partition (6:2:2) **(필수)**



In [ ]:
from sklearn.model_selection import train_test_split

# 1단계: train 60% / (valid + test) 40%
df_train, temp = train_test_split(df, test_size=0.4, random_state=42)

# 2단계: temp를 valid 50% / test 50% → 전체 기준 각 20%
df_valid, df_test = train_test_split(temp, test_size=0.5, random_state=42)

X, Y 분리

In [ ]:
# Train
X_train = df_train.drop('Exited', axis=1)
Y_train = df_train['Exited']

# Valid
X_valid = df_valid.drop('Exited', axis=1)
Y_valid = df_valid['Exited']

# Test
X_test = df_test.drop('Exited', axis=1)
Y_test = df_test['Exited']

One-Hot Encoding

In [ ]:
import pandas as pd

# 범주형 컬럼 선택 (train 기준)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

# One-Hot Encoding
X_train = pd.get_dummies(X_train, drop_first=False, dummy_na=False)
X_valid = pd.get_dummies(X_valid, drop_first=False, dummy_na=False)
X_test  = pd.get_dummies(X_test,  drop_first=False, dummy_na=False)

# valid/test 를 train 컬럼 구조에 맞추기
#    - train에 있고, valid/test에 없는 컬럼 → 0으로 채움
#    - train에 없고, valid/test에 있는 컬럼 → 제거
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test  = X_test.reindex(columns=X_train.columns,  fill_value=0)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test  shape:", X_test.shape)

학습 및 평가

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score)

# 모델 생성
model = DecisionTreeClassifier(random_state=42, max_depth=3)
model.fit(X_train, Y_train)

# X_valid 예측
Y_valid_pred = model.predict(X_valid)
Y_valid_prob = model.predict_proba(X_valid)[:, 1]  # AUC용 확률값

# X_test 예측
Y_test_pred = model.predict(X_test)
Y_test_prob = model.predict_proba(X_test)[:, 1]    # AUC용 확률값

# 평가 함수 정의
def evaluate(Y_actual, Y_pred, Y_prob, dataset_name):
    accuracy  = accuracy_score(Y_actual, Y_pred)
    f1        = f1_score(Y_actual, Y_pred)
    recall    = recall_score(Y_actual, Y_pred)
    precision = precision_score(Y_actual, Y_pred)
    auc       = roc_auc_score(Y_actual, Y_prob)

    print(f"\n=== {dataset_name} 평가 ===")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {'Accuracy': accuracy, 'F1': f1, 'Recall': recall,
            'Precision': precision, 'AUC': auc}

# Validation, Test 평가 실행
valid_scores = evaluate(Y_valid, Y_valid_pred, Y_valid_prob, "Validation")
test_scores  = evaluate(Y_test,  Y_test_pred,  Y_test_prob,  "Test")

# 종합실습 풀이

아래 코드는 과제에서 제공한 6:2:2 분할을 그대로 사용하고, 그 이후 단계에서 Feature Engineering과 모델 최적화를 수행한다.

- `CustomerId`, `Surname`, 원본 날짜 문자열은 모델 입력에서 제외
- 가입일과 기준일의 차이로 거래 기간 파생변수 생성
- 잔액·급여·상품 수·나이·신용점수 조합으로 파생변수 생성
- 결측값은 Train 중앙값 또는 `Unknown`으로 대체
- LightGBM, XGBoost, CatBoost는 Optuna TPE로 각각 60회 탐색
- 앙상블 가중치는 Validation AUC만으로 선택하고 Test는 최종 평가에만 사용

Colab에서는 실행 전에 `bank_churn_train.csv`를 노트북과 같은 작업 경로에 업로드해야 한다.


In [ ]:
# Colab에서 패키지가 없을 때만 주석을 해제하여 실행
# %pip install -q pandas numpy scikit-learn xgboost lightgbm catboost optuna


In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


SEED = 42
DATA_PATH = Path("bank_churn_train.csv")
OUTPUT_DIR = Path("model_optimization_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    x = frame.copy()
    base_date = pd.to_datetime(x["baseDate"], errors="coerce")
    opening_date = pd.to_datetime(x["accountOpeningDate"], errors="coerce")

    x["TenureDays"] = (base_date - opening_date).dt.days
    x["TenureYears"] = x["TenureDays"] / 365.25
    x["BalanceSalaryRatio"] = x["Balance"] / (x["EstimatedSalary"].abs() + 1.0)
    x["BalancePerProduct"] = x["Balance"] / x["NumOfProducts"].clip(lower=1)
    x["SalaryPerProduct"] = x["EstimatedSalary"] / x["NumOfProducts"].clip(lower=1)
    x["CreditScoreAgeRatio"] = x["CreditScore"] / x["Age"].clip(lower=1)
    x["IsZeroBalance"] = (x["Balance"] == 0).astype(int)
    x["AgeSquared"] = x["Age"] ** 2
    x["AgeIsActive"] = x["Age"] * x["IsActiveMember"]
    x["ProductsIsActive"] = x["NumOfProducts"] * x["IsActiveMember"]

    return x.drop(columns=["CustomerId", "Surname", "baseDate", "accountOpeningDate"])


def prepare_one_hot(train, valid, test):
    train = add_features(train)
    valid = add_features(valid)
    test = add_features(test)

    categorical = train.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric = [c for c in train.columns if c not in categorical]

    medians = train[numeric].median()
    train[numeric] = train[numeric].fillna(medians)
    valid[numeric] = valid[numeric].fillna(medians)
    test[numeric] = test[numeric].fillna(medians)

    for col in categorical:
        train[col] = train[col].fillna("Unknown").astype(str)
        valid[col] = valid[col].fillna("Unknown").astype(str)
        test[col] = test[col].fillna("Unknown").astype(str)

    train = pd.get_dummies(train, columns=categorical, drop_first=False, dtype=int)
    valid = pd.get_dummies(valid, columns=categorical, drop_first=False, dtype=int)
    test = pd.get_dummies(test, columns=categorical, drop_first=False, dtype=int)
    valid = valid.reindex(columns=train.columns, fill_value=0)
    test = test.reindex(columns=train.columns, fill_value=0)
    return train, valid, test


def prepare_catboost(train, valid, test):
    train = add_features(train)
    valid = add_features(valid)
    test = add_features(test)
    categorical = train.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric = [c for c in train.columns if c not in categorical]
    medians = train[numeric].median()
    for data in (train, valid, test):
        data[numeric] = data[numeric].fillna(medians)
        for col in categorical:
            data[col] = data[col].fillna("Unknown").astype(str)
    return train, valid, test, categorical


df = pd.read_csv(DATA_PATH, encoding="cp949")

# 과제에서 제공한 필수 분할 코드를 그대로 사용
from sklearn.model_selection import train_test_split

df_train, temp = train_test_split(df, test_size=0.4, random_state=42)
df_valid, df_test = train_test_split(temp, test_size=0.5, random_state=42)

X_train_raw = df_train.drop("Exited", axis=1)
y_train = df_train["Exited"]
X_valid_raw = df_valid.drop("Exited", axis=1)
y_valid = df_valid["Exited"]
X_test_raw = df_test.drop("Exited", axis=1)
y_test = df_test["Exited"]

X_train, X_valid, X_test = prepare_one_hot(X_train_raw, X_valid_raw, X_test_raw)
X_train_cat, X_valid_cat, X_test_cat, cat_cols = prepare_catboost(
    X_train_raw, X_valid_raw, X_test_raw
)

rows = []
predictions = {}


def record(name, model, train_x, valid_x, test_x, note, started):
    valid_prob = model.predict_proba(valid_x)[:, 1]
    test_prob = model.predict_proba(test_x)[:, 1]
    valid_auc = roc_auc_score(y_valid, valid_prob)
    test_auc = roc_auc_score(y_test, test_prob)
    predictions[name] = {"valid": valid_prob, "test": test_prob}
    rows.append(
        {
            "Model": name,
            "Feature Engineering": "Created 10 derived variables; removed ID/name/raw dates; median imputation; categorical encoding",
            "Hyperparameter Tuning": note,
            "AUC_Valid": valid_auc,
            "AUC_Test": test_auc,
            "AVG": (valid_auc + test_auc) / 2,
            "Seconds": time.perf_counter() - started,
        }
    )
    return valid_auc, test_auc


# Baseline-like tree ensemble candidates
baseline_models = {
    "Logistic Regression": Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=700,
        max_depth=8,
        min_samples_leaf=5,
        max_features=0.8,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=700,
        max_depth=9,
        min_samples_leaf=4,
        max_features=0.8,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
}

for name, model in baseline_models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    record(name, model, X_train, X_valid, X_test, "Manual candidate", start)


optuna.logging.set_verbosity(optuna.logging.WARNING)


def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": -1,
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    return roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])


lgbm_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=60)
lgbm_params = {
    **lgbm_study.best_params,
    "subsample_freq": 1,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
}
lgbm_model = LGBMClassifier(**lgbm_params)
start = time.perf_counter()
lgbm_model.fit(X_train, y_train)
record("LightGBM (Optuna)", lgbm_model, X_train, X_valid, X_test, "Optuna TPE, 60 trials", start)


def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 5.0),
        "eval_metric": "auc",
        "random_state": SEED,
        "n_jobs": -1,
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train, verbose=False)
    return roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])


xgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED + 1))
xgb_study.optimize(xgb_objective, n_trials=60)
xgb_params = {
    **xgb_study.best_params,
    "eval_metric": "auc",
    "random_state": SEED,
    "n_jobs": -1,
}
xgb_model = XGBClassifier(**xgb_params)
start = time.perf_counter()
xgb_model.fit(X_train, y_train, verbose=False)
record("XGBoost (Optuna)", xgb_model, X_train, X_valid, X_test, "Optuna TPE, 60 trials", start)


def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 1200, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "depth": trial.suggest_int("depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0),
        "border_count": trial.suggest_categorical("border_count", [32, 64, 128]),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", [None, "Balanced"]),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": SEED,
        "verbose": False,
        "allow_writing_files": False,
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train_cat, y_train, cat_features=cat_cols, verbose=False)
    return roc_auc_score(y_valid, model.predict_proba(X_valid_cat)[:, 1])


cat_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED + 2))
cat_study.optimize(cat_objective, n_trials=60)
cat_params = {
    **cat_study.best_params,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "verbose": False,
    "allow_writing_files": False,
}
cat_model = CatBoostClassifier(**cat_params)
start = time.perf_counter()
cat_model.fit(X_train_cat, y_train, cat_features=cat_cols, verbose=False)
record("CatBoost (Optuna)", cat_model, X_train_cat, X_valid_cat, X_test_cat, "Optuna TPE, 60 trials", start)


# Validation AUC만으로 앙상블 가중치 결정 후 Test는 한 번 평가
ensemble_members = ["LightGBM (Optuna)", "XGBoost (Optuna)", "CatBoost (Optuna)"]
best_ensemble = None
for w_lgbm in np.arange(0.0, 1.01, 0.05):
    for w_xgb in np.arange(0.0, 1.01 - w_lgbm, 0.05):
        w_cat = 1.0 - w_lgbm - w_xgb
        if w_cat < -1e-9:
            continue
        valid_prob = (
            w_lgbm * predictions[ensemble_members[0]]["valid"]
            + w_xgb * predictions[ensemble_members[1]]["valid"]
            + w_cat * predictions[ensemble_members[2]]["valid"]
        )
        auc = roc_auc_score(y_valid, valid_prob)
        if best_ensemble is None or auc > best_ensemble[0]:
            best_ensemble = (auc, w_lgbm, w_xgb, w_cat)

valid_auc, w_lgbm, w_xgb, w_cat = best_ensemble
ensemble_test_prob = (
    w_lgbm * predictions[ensemble_members[0]]["test"]
    + w_xgb * predictions[ensemble_members[1]]["test"]
    + w_cat * predictions[ensemble_members[2]]["test"]
)
test_auc = roc_auc_score(y_test, ensemble_test_prob)
rows.append(
    {
        "Model": "Weighted Soft Voting Ensemble",
        "Feature Engineering": "Same engineered feature set; CatBoost native categories + one-hot tree models",
        "Hyperparameter Tuning": f"Validation-only weight search: LGBM={w_lgbm:.2f}, XGB={w_xgb:.2f}, CatBoost={w_cat:.2f}",
        "AUC_Valid": valid_auc,
        "AUC_Test": test_auc,
        "AVG": (valid_auc + test_auc) / 2,
        "Seconds": np.nan,
    }
)

results = pd.DataFrame(rows).sort_values("AVG", ascending=False).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False)

payload = {
    "data": {
        "total_rows": int(len(df)),
        "train_rows": int(len(df_train)),
        "valid_rows": int(len(df_valid)),
        "test_rows": int(len(df_test)),
        "train_positive_rate": float(y_train.mean()),
        "valid_positive_rate": float(y_valid.mean()),
        "test_positive_rate": float(y_test.mean()),
    },
    "lgbm_best_params": lgbm_params,
    "xgb_best_params": xgb_params,
    "catboost_best_params": cat_params,
    "ensemble_weights": {"LightGBM": w_lgbm, "XGBoost": w_xgb, "CatBoost": w_cat},
    "results": results.replace({np.nan: None}).to_dict(orient="records"),
}
(OUTPUT_DIR / "optimization_summary.json").write_text(
    json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(results.to_string(index=False))
print(json.dumps(payload, ensure_ascii=False, indent=2))


## 최종 결과

| 모델 | Validation AUC | Test AUC | 평균 AUC |
|:--|--:|--:|--:|
| **Weighted Soft Voting Ensemble** | **0.8745** | **0.8535** | **0.8640** |
| CatBoost (Optuna) | 0.8733 | 0.8505 | 0.8619 |
| LightGBM (Optuna) | 0.8690 | 0.8535 | 0.8613 |
| XGBoost (Optuna) | 0.8660 | 0.8490 | 0.8575 |
| Extra Trees | 0.8406 | 0.8645 | 0.8526 |
| Random Forest | 0.8298 | 0.8166 | 0.8232 |
| Logistic Regression | 0.7820 | 0.7954 | 0.7887 |

최종 모델은 LightGBM 0.25, XGBoost 0.25, CatBoost 0.50의 예측 확률을 가중 평균한 Soft Voting 앙상블이다. Validation AUC만 사용해 가중치를 정했으며 Test 데이터는 모델·가중치 선택에 사용하지 않았다.
